<a href="https://colab.research.google.com/github/imranDv1/Blog-Imran/blob/main/plant_disease_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plant Disease CNN (from scratch, PyTorch)
Runtime > Change runtime type > **T4 GPU**, then run all cells top to bottom.
Trains a CNN you define yourself (no pretrained weights) on PlantVillage (38 classes).

In [7]:
# Install and imports
import os, json, time, random, subprocess, sys
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU")

Device: cuda


In [8]:
# Download PlantVillage (public dataset, no Kaggle login needed)
root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
DATA_DIR = None
for dp, dn, fn in os.walk(root):
    if os.path.basename(dp).lower() == "color":
        DATA_DIR = dp
        break
assert DATA_DIR, "Could not find the 'color' folder"
print("Using:", DATA_DIR)

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Using: /kaggle/input/plantvillage-dataset/plantvillage dataset/color


In [9]:
# Data: transforms, stratified 80/10/10 split, loaders
IMG = 128
BATCH = 64
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # just normalization, not pretrained weights

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

ds_train_full = datasets.ImageFolder(DATA_DIR, transform=train_tf)
ds_eval_full = datasets.ImageFolder(DATA_DIR, transform=eval_tf)
classes = ds_train_full.classes
targets = np.array(ds_train_full.targets)
print(f"{len(ds_train_full)} images, {len(classes)} classes")

idx = np.arange(len(targets))
train_idx, tmp_idx = train_test_split(idx, test_size=0.2, stratify=targets, random_state=SEED)
val_idx, test_idx = train_test_split(tmp_idx, test_size=0.5, stratify=targets[tmp_idx], random_state=SEED)

train_loader = DataLoader(Subset(ds_train_full, train_idx), batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(Subset(ds_eval_full, val_idx), batch_size=BATCH, shuffle=False,
                        num_workers=2, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(Subset(ds_eval_full, test_idx), batch_size=BATCH, shuffle=False,
                         num_workers=2, pin_memory=True)
print(f"train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}")

54305 images, 38 classes
train 43444 | val 5430 | test 5431


In [10]:
# The model (written by us, random init)
def block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )

class PlantCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            block(3, 32), block(32, 64), block(64, 128), block(128, 256), block(256, 256)
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.head(self.features(x))

model = PlantCNN(len(classes)).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

Parameters: 2.36M


In [11]:
# Train
EPOCHS = 15
use_amp = device == "cuda"
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, epochs=EPOCHS, steps_per_epoch=len(train_loader))
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def evaluate(loader):
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device, enabled=use_amp):
                out = model(x)
                loss = criterion(out, y)
            loss_sum += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return loss_sum / total, correct / total

best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    run_loss = run_correct = run_total = 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device, enabled=use_amp):
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        run_loss += loss.item() * y.size(0)
        run_correct += (out.argmax(1) == y).sum().item()
        run_total += y.size(0)
    val_loss, val_acc = evaluate(val_loader)
    print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {run_loss/run_total:.3f} acc {run_correct/run_total:.3f} "
          f"| val loss {val_loss:.3f} acc {val_acc:.3f} | {time.time()-t0:.0f}s")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best.pt")
print("Best val acc:", best_acc)

Epoch  1/15 | train loss 2.013 acc 0.559 | val loss 1.401 acc 0.750 | 240s
Epoch  2/15 | train loss 1.309 acc 0.797 | val loss 1.523 acc 0.701 | 166s
Epoch  3/15 | train loss 1.140 acc 0.863 | val loss 1.107 acc 0.867 | 166s
Epoch  4/15 | train loss 1.037 acc 0.901 | val loss 0.946 acc 0.930 | 170s
Epoch  5/15 | train loss 0.970 acc 0.923 | val loss 0.898 acc 0.941 | 170s
Epoch  6/15 | train loss 0.919 acc 0.942 | val loss 0.908 acc 0.931 | 167s
Epoch  7/15 | train loss 0.886 acc 0.951 | val loss 0.835 acc 0.960 | 174s
Epoch  8/15 | train loss 0.855 acc 0.963 | val loss 0.817 acc 0.965 | 171s
Epoch  9/15 | train loss 0.831 acc 0.969 | val loss 0.826 acc 0.963 | 168s
Epoch 10/15 | train loss 0.807 acc 0.977 | val loss 0.756 acc 0.984 | 168s
Epoch 11/15 | train loss 0.787 acc 0.981 | val loss 0.735 acc 0.990 | 168s
Epoch 12/15 | train loss 0.769 acc 0.986 | val loss 0.726 acc 0.993 | 169s
Epoch 13/15 | train loss 0.754 acc 0.990 | val loss 0.713 acc 0.995 | 169s
Epoch 14/15 | train loss 

In [12]:
import time
elapsed_first_epoch = time.time() - t0
remaining_epochs = EPOCHS - epoch
estimated_remaining_seconds = elapsed_first_epoch * remaining_epochs

m, s = divmod(int(estimated_remaining_seconds), 60)
h, m = divmod(m, 60)

print(f"Current epoch: {epoch}/{EPOCHS}")
print(f"Time taken for first epoch: {elapsed_first_epoch:.1f} seconds")
print(f"Remaining epochs: {remaining_epochs}")
print(f"Estimated time remaining: {h}h {m}m {s}s")

Current epoch: 15/15
Time taken for first epoch: 171.7 seconds
Remaining epochs: 0
Estimated time remaining: 0h 0m 0s


In [13]:
# Final test on data the model never saw
model.load_state_dict(torch.load("best.pt", map_location=device))
test_loss, test_acc = evaluate(test_loader)
print(f"TEST accuracy: {test_acc:.4f}")

TEST accuracy: 0.9950


In [14]:
# Save model + labels
torch.save({
    "state_dict": model.state_dict(),
    "classes": classes,
    "img_size": IMG,
    "mean": mean,
    "std": std,
}, "plant_cnn.pt")
with open("classes.json", "w") as f:
    json.dump(classes, f, indent=2)
print("Saved plant_cnn.pt and classes.json")

Saved plant_cnn.pt and classes.json


In [15]:
# Predict on your own image
def predict(path, threshold=0.6, topk=3):
    img = Image.open(path).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)[0]
    conf, ids = probs.topk(topk)
    results = []
    for c, i in zip(conf.tolist(), ids.tolist()):
        plant, _, disease = classes[i].partition("___")
        results.append({
            "plant": plant.replace("_", " "),
            "disease": disease.replace("_", " "),
            "healthy": disease.lower() == "healthy",
            "confidence": round(c, 3),
        })
    if results[0]["confidence"] < threshold:
        print("Not sure. Please upload a clearer photo of a single leaf.")
    return results

try:
    from google.colab import files
    up = files.upload()
    for name in up:
        print(name)
        for r in predict(name):
            print("  ", r)
except ImportError:
    print("Not running in Colab: call predict('path/to/leaf.jpg') manually.")

Saving 00fee259-67b7-4dd7-8b36-12503bbdba14___RS_HL 2681.JPG to 00fee259-67b7-4dd7-8b36-12503bbdba14___RS_HL 2681.JPG
00fee259-67b7-4dd7-8b36-12503bbdba14___RS_HL 2681.JPG
   {'plant': 'Blueberry', 'disease': 'healthy', 'healthy': True, 'confidence': 0.916}
   {'plant': 'Potato', 'disease': 'Late blight', 'healthy': False, 'confidence': 0.007}
   {'plant': 'Potato', 'disease': 'healthy', 'healthy': True, 'confidence': 0.005}


In [16]:
# Download the trained model to your computer
try:
    from google.colab import files
    files.download("plant_cnn.pt")
    files.download("classes.json")
except ImportError:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Test API (base64 in, JSON out)
Run the cells below **after** training has finished and `plant_cnn.pt` is saved.

In [17]:
%%writefile server.py
"""Plant disease API.

Run locally:   pip install fastapi uvicorn python-multipart pillow torch torchvision
               uvicorn server:app --reload --port 8000
Needs plant_cnn.pt (from the training notebook) next to this file.
Optional: treatments.json = {"Tomato___Late_blight": {"description": "...", "symptoms": [...], "treatment": [...]}}
"""
import base64, binascii, io, json, os

import torch
import torch.nn as nn
import torch.nn.functional as F
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image, UnidentifiedImageError
from pydantic import BaseModel
from torchvision import transforms

MODEL_PATH = os.getenv("MODEL_PATH", "plant_cnn.pt")
TREATMENTS_PATH = os.getenv("TREATMENTS_PATH", "treatments.json")
CONF_THRESHOLD = float(os.getenv("CONF_THRESHOLD", "0.6"))
MAX_BYTES = 10 * 1024 * 1024
device = "cuda" if torch.cuda.is_available() else "cpu"


def block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class PlantCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            block(3, 32), block(32, 64), block(64, 128), block(128, 256), block(256, 256)
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(self.features(x))


ckpt = torch.load(MODEL_PATH, map_location=device)
classes = ckpt["classes"]
model = PlantCNN(len(classes)).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

tf = transforms.Compose([
    transforms.Resize((ckpt["img_size"], ckpt["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(ckpt["mean"], ckpt["std"]),
])

treatments = {}
if os.path.exists(TREATMENTS_PATH):
    with open(TREATMENTS_PATH, encoding="utf-8") as f:
        treatments = json.load(f)

app = FastAPI(title="Plant Disease API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


class Base64Request(BaseModel):
    image: str  # raw base64, or a data URL like "data:image/jpeg;base64,...."


def decode_base64_image(data: str) -> Image.Image:
    if data.startswith("data:"):
        data = data.split(",", 1)[-1]  # drop "data:image/png;base64,"
    try:
        raw = base64.b64decode(data, validate=False)
    except (binascii.Error, ValueError):
        raise HTTPException(400, "Invalid base64 string")
    return bytes_to_image(raw)


def bytes_to_image(raw: bytes) -> Image.Image:
    if len(raw) > MAX_BYTES:
        raise HTTPException(413, "Image too large (max 10MB)")
    try:
        return Image.open(io.BytesIO(raw)).convert("RGB")
    except (UnidentifiedImageError, OSError):
        raise HTTPException(400, "Could not read an image from the data")


def run_model(img: Image.Image, topk: int = 3) -> dict:
    x = tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)[0]
    conf, ids = probs.topk(min(topk, len(classes)))
    top = []
    for c, i in zip(conf.tolist(), ids.tolist()):
        plant, _, disease = classes[i].partition("___")
        top.append({
            "label": classes[i],
            "plant": plant.replace("_", " "),
            "disease": disease.replace("_", " "),
            "healthy": disease.lower() == "healthy",
            "confidence": round(c, 4),
        })
    best = top[0]
    info = treatments.get(best["label"], {})
    return {
        "recognized": best["confidence"] >= CONF_THRESHOLD,
        "message": None if best["confidence"] >= CONF_THRESHOLD
        else "Not sure. Please upload a clearer photo of a single leaf.",
        **best,
        "description": info.get("description"),
        "symptoms": info.get("symptoms"),
        "treatment": info.get("treatment"),
        "top_predictions": top,
    }


@app.get("/health")
def health():
    return {"status": "ok", "classes": len(classes), "device": device}


@app.post("/predict")
def predict_base64(req: Base64Request):
    return run_model(decode_base64_image(req.image))


@app.post("/predict-file")
async def predict_file(file: UploadFile = File(...)):
    return run_model(bytes_to_image(await file.read()))

Writing server.py


In [18]:
# Start the API in the background
!pip install -q fastapi uvicorn python-multipart
import subprocess, sys, time, requests
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
                       stdout=open("api.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(requests.get("http://127.0.0.1:8000/health", timeout=2).json())
        break
    except Exception:
        time.sleep(1)
else:
    print("API did not start. Log:\n", open("api.log").read())

{'status': 'ok', 'classes': 38, 'device': 'cuda'}


In [19]:
# Test 1: one random test image sent as base64
import base64, random, json, requests
path, label_id = ds_eval_full.samples[int(random.choice(test_idx))]
with open(path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()
r = requests.post("http://127.0.0.1:8000/predict", json={"image": b64})
print("True label:", classes[label_id])
print(json.dumps(r.json(), indent=2))

True label: Orange___Haunglongbing_(Citrus_greening)
{
  "recognized": true,
  "message": null,
  "label": "Orange___Haunglongbing_(Citrus_greening)",
  "plant": "Orange",
  "disease": "Haunglongbing (Citrus greening)",
  "healthy": false,
  "confidence": 0.9072,
  "description": null,
  "symptoms": null,
  "treatment": null,
  "top_predictions": [
    {
      "label": "Orange___Haunglongbing_(Citrus_greening)",
      "plant": "Orange",
      "disease": "Haunglongbing (Citrus greening)",
      "healthy": false,
      "confidence": 0.9072
    },
    {
      "label": "Soybean___healthy",
      "plant": "Soybean",
      "disease": "healthy",
      "healthy": true,
      "confidence": 0.0049
    },
    {
      "label": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
      "plant": "Tomato",
      "disease": "Tomato Yellow Leaf Curl Virus",
      "healthy": false,
      "confidence": 0.0045
    }
  ]
}


In [20]:
# Test 2: accuracy measured THROUGH the API on 100 unseen test images
n, ok = 100, 0
for i in random.sample([int(i) for i in test_idx], n):
    p, y = ds_eval_full.samples[i]
    with open(p, "rb") as f:
        b = base64.b64encode(f.read()).decode()
    pred = requests.post("http://127.0.0.1:8000/predict", json={"image": b}).json()
    ok += pred["label"] == classes[y]
print(f"API accuracy on {n} unseen images: {ok}/{n}")

API accuracy on 100 unseen images: 100/100


In [23]:
# Test 3: your own image -> base64 -> API
from google.colab import files
up = files.upload()
for name, data in up.items():
    b64 = base64.b64encode(data).decode()
    r = requests.post("http://127.0.0.1:8000/predict", json={"image": b64})
    print(name, json.dumps(r.json(), indent=2))

Saving image_processing20220902-1323614-nlh319.jpg to image_processing20220902-1323614-nlh319.jpg
image_processing20220902-1323614-nlh319.jpg {
  "recognized": false,
  "message": "Not sure. Please upload a clearer photo of a single leaf.",
  "label": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
  "plant": "Tomato",
  "disease": "Tomato Yellow Leaf Curl Virus",
  "healthy": false,
  "confidence": 0.2885,
  "description": null,
  "symptoms": null,
  "treatment": null,
  "top_predictions": [
    {
      "label": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
      "plant": "Tomato",
      "disease": "Tomato Yellow Leaf Curl Virus",
      "healthy": false,
      "confidence": 0.2885
    },
    {
      "label": "Squash___Powdery_mildew",
      "plant": "Squash",
      "disease": "Powdery mildew",
      "healthy": false,
      "confidence": 0.135
    },
    {
      "label": "Orange___Haunglongbing_(Citrus_greening)",
      "plant": "Orange",
      "disease": "Haunglongbing (Citrus greening)",
     